# KSE EP-OpInf Tests

This notebook tests the Kuramoto–Sivashinsky ROMs trained in `06_KSE_EP-OpInf.ipynb` on new initial conditions

$$
u(\omega,0) = a\cos\left( \frac{2\pi \omega}{L} \right) + b\cos\left( \frac{4\pi \omega}{L} \right):
$$

- **Test 1 (interpolation):** $(a, b)$ drawn uniformly at random inside the training range $[0.8, 1.2]\times[0.2, 0.6]$.
- **Test 2 (extrapolation):** $(a, b)$ drawn outside the training range: half of the samples in $[0, 0.79]\times[0, 0.19]$ and half in $[1.21, 2]\times[0.61, 0.8]$.

For each test we compare the average normalized autocorrelation of the full-order model (FOM) and the ROMs (standard OpInf, intrusive, EPHEC-OpInf), the short-time relative state error of all ROMs, and the predicted flow fields.

**Prerequisite:** run notebook 06 first, in the same directory. This notebook reads

| file | from notebook 06 |
|---|---|
| `data/kse_data_L22.jld2` | simulation settings (incl. `PAPER_SCALE`), POD basis, reduced orders, FOM operators |
| `data/kse_operators_L22.jld2` | ROM operators |

The demo/paper setting `PAPER_SCALE` is **taken from notebook 06's saved settings**, so the two notebooks always use consistent sizes. With `PAPER_SCALE = false` each test uses 10 random initial conditions; the paper uses 50.

This notebook writes `data/kse_test_ics_L22.jld2` (the random test parameters; it is reused when it exists and matches the current number of test cases) and `data/kse_test_results_L22.jld2` (the results).

In [ ]:
using DataFrames
using FileIO
using JLD2
using LaTeXStrings
using LinearAlgebra
using Plots
using Plots.PlotMeasures
using ProgressMeter
using Random
using SparseArrays
using Statistics
using StatsBase
using UniqueKronecker
using PolynomialModelReductionDataset: KuramotoSivashinskyModel

using LiftAndLearn
const LnL = LiftAndLearn
;

In [ ]:
# Files
datafile = "data/kse_data_L22.jld2"
opfile = "data/kse_operators_L22.jld2"
testresultfile = "data/kse_test_results_L22.jld2"
testicfile = "data/kse_test_ics_L22.jld2"

for f in (datafile, opfile)
    isfile(f) || error("Required file '$(f)' not found. Run notebook 06_KSE_EP-OpInf.ipynb first " *
                       "(in the same directory) to generate the training data and the ROM operators.")
end
mkpath("figures")

DATA = load(datafile)
OPS = load(opfile)
SETTINGS = DATA["settings"]
;

In [ ]:
# Demo or paper scale: set by notebook 06 (see its settings cell)
const PAPER_SCALE = SETTINGS["PAPER_SCALE"]

# Number of random test initial conditions per test
num_test_ic = PAPER_SCALE ? 50 : 10

# Short-time horizon for the state errors (time units)
T_SHORT = 20.0

# Settings of the simulations (identical to notebook 06)
Ω = SETTINGS["Omega"]
Nx = SETTINGS["Nx"]
T_END = SETTINGS["T_END"]
EVAL_STRIDE = SETTINGS["EVAL_STRIDE"]  # time steps between the samples used for errors, lags and plots
SAVE_STRIDE = SETTINGS["SAVE_STRIDE"]  # time steps between the stored FOM samples

KSE = KuramotoSivashinskyModel(
    spatial_domain=Ω, time_domain=(0.0, T_END), Δx=(Ω[2] - Ω[1]) / Nx, Δt=SETTINGS["dt"],
    diffusion_coeffs=SETTINGS["mu"], BC=:periodic, conservation_type=:EP,
)
@assert KSE.spatial_dim == size(DATA["Vr"][1], 1) "grid does not match the data of notebook 06"
nsteps = KSE.time_dim - 1

# Training initial-condition parameters and reduced orders
ic_a = SETTINGS["ic_a"]
ic_b = SETTINGS["ic_b"]
ro = DATA["ro"]
println("PAPER_SCALE = $(PAPER_SCALE), reduced orders = $(ro), test cases per test = $(num_test_ic)")
;

In [ ]:
# Parameterized function for the initial condition
L = KSE.spatial_domain[2] - KSE.spatial_domain[1]  # length of the domain
u0 = (a,b) -> a * cos.((2*π*KSE.xspan)/L) .+ b * cos.((4*π*KSE.xspan)/L)  # initial condition
;

## Test initial conditions

The random test parameters are drawn with a fixed seed and saved to `data/kse_test_ics_L22.jld2`. An existing file is reused if it holds the same number of test cases.

In [ ]:
function generate_test_ics(num_test_ic; rng=MersenneTwister(1234))
    # Test 1: inside the training range
    ic_a_rand_in = (maximum(ic_a) - minimum(ic_a)) .* rand(rng, num_test_ic) .+ minimum(ic_a)
    ic_b_rand_in = (maximum(ic_b) - minimum(ic_b)) .* rand(rng, num_test_ic) .+ minimum(ic_b)

    # Test 2: outside the training range (half below, half above)
    ic_a_out = [0.0, 2.0]
    ic_b_out = [0.0, 0.8]
    ϵ = 1e-2
    half_num_test_ic = num_test_ic ÷ 2
    ic_a_rand_out = ((minimum(ic_a) - ϵ) - ic_a_out[1]) .* rand(rng, half_num_test_ic) .+ ic_a_out[1]
    append!(ic_a_rand_out, (ic_a_out[2] - (maximum(ic_a) + ϵ)) .* rand(rng, num_test_ic - half_num_test_ic) .+ (maximum(ic_a) + ϵ))
    ic_b_rand_out = ((minimum(ic_b) - ϵ) - ic_b_out[1]) .* rand(rng, half_num_test_ic) .+ ic_b_out[1]
    append!(ic_b_rand_out, (ic_b_out[2] - (maximum(ic_b) + ϵ)) .* rand(rng, num_test_ic - half_num_test_ic) .+ (maximum(ic_b) + ϵ))

    return Dict(
        "num_test_ic" => num_test_ic,
        "ic_a_rand_in" => ic_a_rand_in,
        "ic_b_rand_in" => ic_b_rand_in,
        "ic_a_rand_out" => ic_a_rand_out,
        "ic_b_rand_out" => ic_b_rand_out,
    )
end

ICs = isfile(testicfile) ? load(testicfile) : Dict{String, Any}()
if get(ICs, "num_test_ic", 0) != num_test_ic
    @info "Generating $(num_test_ic) test initial conditions per test and saving them to $(testicfile)"
    ICs = generate_test_ics(num_test_ic)
    save(testicfile, ICs)
end
ic_a_rand_in = ICs["ic_a_rand_in"]
ic_b_rand_in = ICs["ic_b_rand_in"]
ic_a_rand_out = ICs["ic_a_rand_out"]
ic_b_rand_out = ICs["ic_b_rand_out"]
;

## Simulation tools

The full-order test trajectories are computed with the same fast CNAB stepper as in notebook 06. It runs the scheme of `KSE.integrate_model`, but evaluates the quadratic term from the nonzeros of the sparse operator. The ROMs are simulated with `KSE.integrate_model`.

In [ ]:
"""
    CNABStepper(A, F, Δt)

Crank–Nicolson (linear part) / Adams–Bashforth (quadratic part) time stepper for
`ẋ = A x + F (x ⊘ x)`, i.e. the scheme of `KSE.integrate_model`, evaluating `F (x ⊘ x)`
from the nonzeros of `F`.
"""
struct CNABStepper
    Δt::Float64
    Mfac                  # factorization of I - Δt/2 A
    P                     # I + Δt/2 A
    qrow::Vector{Int}     # nonzeros of F: F[qrow, col] multiplies x[qi] * x[qj]
    qi::Vector{Int}
    qj::Vector{Int}
    qval::Vector{Float64}
end

function CNABStepper(A, F, Δt)
    n = size(A, 1)
    pairs = [(i, j) for i in 1:n for j in i:n]  # column ordering of x ⊘ x
    rows, cols, vals = findnz(sparse(F))
    if issparse(A)
        Mfac = lu(sparse(1.0I(n) - Δt/2 * A))
        P = sparse(1.0I(n) + Δt/2 * A)
    else
        Mfac = lu(Matrix(1.0I(n) - Δt/2 * A))
        P = Matrix(1.0I(n) + Δt/2 * A)
    end
    return CNABStepper(Δt, Mfac, P, rows, first.(pairs[cols]), last.(pairs[cols]), vals)
end

function quad!(f, S::CNABStepper, u)
    fill!(f, 0.0)
    @inbounds for p in eachindex(S.qval)
        f[S.qrow[p]] += S.qval[p] * u[S.qi[p]] * u[S.qj[p]]
    end
    return f
end

"""
    cnab_solve(S, u0, nsteps; keep=0:nsteps)

Integrate `nsteps` time steps from `u0` and return the states at the time steps listed in the
sorted vector `keep` (step 0 is the initial condition) as the columns of a matrix.
"""
function cnab_solve(S::CNABStepper, u0::AbstractVector, nsteps::Integer; keep=0:nsteps)
    @assert issorted(keep) && first(keep) >= 0 && last(keep) <= nsteps
    X = zeros(length(u0), length(keep))
    u = Vector{Float64}(u0)
    f = zero(u); fold = zero(u)
    c = 1
    while c <= length(keep) && keep[c] == 0
        X[:, c] = u; c += 1
    end
    for k in 1:nsteps
        quad!(f, S, u)
        if k == 1  # forward Euler for the quadratic term in the first step (as in KSE.integrate_model)
            rhs = S.P * u .+ S.Δt .* f
        else
            rhs = S.P * u .+ (1.5 * S.Δt) .* f .- (0.5 * S.Δt) .* fold
        end
        u = S.Mfac \ rhs
        f, fold = fold, f
        while c <= length(keep) && keep[c] == k
            X[:, c] = u; c += 1
        end
    end
    return X
end

"""Simulate the r-dimensional ROM obtained by truncating the operators `op` from the initial condition `x0`."""
function simulate_rom(model, op, r, x0; tspan=model.tspan)
    Ar = op.A[1:r, 1:r]
    Fr = UniqueKronecker.extractF(op.A2u, r)
    return model.integrate_model(tspan, x0; linear_matrix=Ar, quadratic_matrix=Fr,
                                 system_input=false, const_stepsize=true)
end
;

## Autocorrelation Functions and Test Procedure

For every test initial condition we simulate the FOM and all ROMs, and compute

- the mean (over the grid) normalized autocorrelation, for the FOM and the standard OpInf, intrusive and EPHEC ROMs, with lags spaced `EVAL_STRIDE` time steps apart up to half the time horizon, as in notebook 06;
- the relative state error over the short horizon $t \in [0, T_{\text{short}}]$ for all five ROMs. Over the full horizon every ROM of this chaotic system decorrelates from the FOM.

In [ ]:
function tmean_autocorr(X::AbstractArray, lags::AbstractVector)
    N = size(X, 1)
    Cx = zeros(N, length(lags))
    for i in 1:N  # normalized autocorrelation
        Cx[i, :] = autocor(X[i, :], lags)
    end
    return vec(mean(Cx, dims=1))
end

rom_keys = (:LS, :int, :ephec, :epsic, :epp)  # ROMs for the short-time state errors
ac_keys = (:LS, :int, :ephec)                  # ROMs for the autocorrelation
rom_ops = Dict(:LS => OPS["op_LS"], :int => OPS["op_int"], :ephec => OPS["op_ephec"],
               :epsic => OPS["op_epsic"], :epp => OPS["op_epp"])

# Time lags in units of the stored samples (spaced EVAL_STRIDE time steps apart, up to T_END/2)
lags = 0:(EVAL_STRIDE ÷ SAVE_STRIDE):((KSE.time_dim ÷ 2) ÷ SAVE_STRIDE)
lags_t = collect(lags) .* SAVE_STRIDE .* KSE.Δt
k_short = round(Int, T_SHORT / KSE.Δt)  # number of time steps of the short horizon

"""
Run one test: simulate the FOM and the ROMs for the initial-condition parameters (a_vals, b_vals) and
return the mean autocorrelations, the mean autocorrelation errors and the mean short-time state errors.
"""
function run_test(a_vals, b_vals)
    n = length(a_vals)
    A = DATA["op_fom_tr"][1].A
    F = DATA["op_fom_tr"][1].A2u
    Vr_all = DATA["Vr"][1]

    ac_fom = Vector{Vector{Float64}}(undef, n)
    ac_rom = Dict(key => Vector{Matrix{Float64}}(undef, n) for key in ac_keys)
    se_rom = Dict(key => Matrix{Float64}(undef, length(ro), n) for key in rom_keys)

    prog = Progress(n)
    Threads.@threads for j in 1:n
        IC = u0(a_vals[j], b_vals[j])
        S = CNABStepper(A, F, KSE.Δt)  # one stepper per task (the LU factorization is not thread-safe)
        Xfom = cnab_solve(S, IC, nsteps; keep=0:SAVE_STRIDE:nsteps)
        ac_fom[j] = tmean_autocorr(Xfom, lags)
        Xfom_short = Xfom[:, 1:(EVAL_STRIDE ÷ SAVE_STRIDE):(k_short ÷ SAVE_STRIDE + 1)]

        for key in rom_keys
            key in ac_keys && (ac_rom[key][j] = Matrix{Float64}(undef, length(lags), length(ro)))
            for (ri, r) in enumerate(ro)
                Vr = Vr_all[:, 1:r]
                tspan = key in ac_keys ? KSE.tspan : KSE.tspan[1:k_short+1]
                X = simulate_rom(KSE, rom_ops[key][1], r, Vr' * IC; tspan=tspan)
                se_rom[key][ri, j] = LnL.rel_state_error(Xfom_short, X[:, 1:EVAL_STRIDE:k_short+1], Vr)
                if key in ac_keys
                    ac_rom[key][j][:, ri] = tmean_autocorr(Vr * X[:, 1:SAVE_STRIDE:end], lags)
                end
            end
        end
        next!(prog)
    end

    # Averages over all test initial conditions
    AC = Dict{Symbol, Array{Float64}}(:fom => mean(ac_fom))
    AC_ERR = Dict{Symbol, Matrix{Float64}}()
    for key in ac_keys
        AC[key] = mean(ac_rom[key])
        AC_ERR[key] = reshape([mean(norm(ac_rom[key][j][:, ri] - ac_fom[j]) / norm(ac_fom[j]) for j in 1:n) for ri in eachindex(ro)], :, 1)
    end
    SE = Dict{Symbol, Matrix{Float64}}(key => mean(se_rom[key], dims=2) for key in rom_keys)
    return AC, AC_ERR, SE
end

TEST_RES = Dict{String, Any}("AC_lags" => lags_t, "T_SHORT" => T_SHORT, "num_test_ic" => num_test_ic)
;

## Test 1

In [ ]:
TEST_RES["test1_AC"], TEST_RES["test1_AC_ERR"], TEST_RES["test1_state_err_short"] = run_test(ic_a_rand_in, ic_b_rand_in)
DataFrame(r=ro; (Symbol(key, "_AC_err") => vec(TEST_RES["test1_AC_ERR"][key]) for key in ac_keys)...)

In [ ]:
DataFrame(r=ro; (Symbol(key, "_state_err_t≤$(Int(T_SHORT))") => vec(TEST_RES["test1_state_err_short"][key]) for key in rom_keys)...)

In [ ]:
function plot_ac4(AC, title)
    ri_plot = round.(Int, range(1, length(ro), length=4))  # four of the reduced orders
    p = plot(layout=@layout([grid(2,2)]), size=(1000, 550), plot_title=title)
    for (plot_id, ri) in enumerate(ri_plot)
        plot!(p[plot_id], lags_t, AC[:LS][:, ri], c=:black, label="opinf")
        plot!(p[plot_id], lags_t, AC[:int][:, ri], c=:orange, label="intrusive")
        plot!(p[plot_id], lags_t, AC[:ephec][:, ri], c=:blue, ls=:dash, lw=2, label="ep-opinf")
        plot!(p[plot_id], lags_t, AC[:fom], c=:green, lw=1, ls=:dash, label="full")
        plot!(p[plot_id], fontfamily="Computer Modern", guidefontsize=12, tickfontsize=13)
        plot!(p[plot_id], title="r = $(ro[ri])", titlefontsize=14, titlefontfamily="Computer Modern")
        yticks!(p[plot_id], [-0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
        if plot_id == 4
            plot!(p[plot_id], legend=:topright, legend_column=2, legendfontsize=11)
        else
            plot!(p[plot_id], legend=false)
        end
    end
    xlabel!(p[3], "time lag"); xlabel!(p[4], "time lag")
    ylabel!(p[1], "avg normalized autocorr."); ylabel!(p[3], "avg normalized autocorr.")
    plot!(p[1], left_margin=5mm)
    plot!(p[3], bottom_margin=5mm, left_margin=5mm)
    plot!(p[4], bottom_margin=5mm)
    return p
end

function plot_ac_err(AC_ERR, title)
    p = plot(ro, AC_ERR[:LS], c=:black, marker=(:circle, 3.5, :black), label="OpInf")
    plot!(ro, AC_ERR[:int], c=:orange, marker=(:cross, 8, :orange), markerstrokewidth=2.5, label="Intrusive")
    plot!(ro, AC_ERR[:ephec], c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), ls=:dash, lw=2, label="EP-OpInf")
    plot!(majorgrid=true, legend=:topright)
    xlabel!("reduced model dimension " * L" r")
    ylabel!("avg normalized autocorrelation error")
    title!(title, titlefontsize=16)
    xticks!(ro)
    plot!(left_margin=2mm)
    plot!(fontfamily="Computer Modern", guidefontsize=13, tickfontsize=13,  legendfontsize=13)
    return p
end

function plot_short_err(SE, title)
    p = plot(ro, SE[:LS], c=:black, marker=(:circle, 3.5, :black), label="OpInf")
    plot!(ro, SE[:int], c=:orange, marker=(:cross, 8, :orange), markerstrokewidth=2.5, label="Intrusive")
    plot!(ro, SE[:ephec], c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), ls=:dash, lw=2, label="EPHEC-OpInf")
    plot!(ro, SE[:epsic], c=:purple, markerstrokecolor=:purple, marker=(:dtriangle, 5), ls=:dot, label="EPSIC-OpInf")
    plot!(ro, SE[:epp], c=:red, markerstrokecolor=:red, marker=(:star, 4), lw=1, ls=:dash, label="EPP-OpInf")
    plot!(majorgrid=true, legend=:topright, yscale=:log10)
    xlabel!("reduced model dimension " * L" r")
    ylabel!("avg relative state error, " * L"t \leq %$(Int(T_SHORT))")
    title!(title, titlefontsize=16)
    xticks!(ro)
    plot!(fontfamily="Computer Modern", guidefontsize=13, tickfontsize=13, legendfontsize=12)
    return p
end
;

In [ ]:
p = plot_ac4(TEST_RES["test1_AC"], "Test 1: interpolation")
savefig(p, "figures/kse_test1_ac4.pdf")
p

In [ ]:
p = plot_ac_err(TEST_RES["test1_AC_ERR"], "Test 1: Interpolation")
savefig(p, "figures/kse_test1_ac_err.pdf")
p

In [ ]:
plot_short_err(TEST_RES["test1_state_err_short"], "Test 1: Interpolation")

### Flow field of one test case

In [ ]:
function plot_flow_fields(a, b, title)
    r = ro[end]
    ic = u0(a, b)
    Vr = DATA["Vr"][1][:, 1:r]
    t_plot = KSE.tspan[1:EVAL_STRIDE:end]

    S = CNABStepper(DATA["op_fom_tr"][1].A, DATA["op_fom_tr"][1].A2u, KSE.Δt)
    U_fom = cnab_solve(S, ic, nsteps; keep=0:EVAL_STRIDE:nsteps)
    U_int = Vr * simulate_rom(KSE, OPS["op_int"][1], r, Vr' * ic)[:, 1:EVAL_STRIDE:end]
    U_LS = Vr * simulate_rom(KSE, OPS["op_LS"][1], r, Vr' * ic)[:, 1:EVAL_STRIDE:end]
    U_ephec = Vr * simulate_rom(KSE, OPS["op_ephec"][1], r, Vr' * ic)[:, 1:EVAL_STRIDE:end]

    field(U; kw...) = plot(contourf(t_plot, KSE.xspan, U, lw=0, color=:inferno);
                           colorbar_ticks=(-3:3), yticks=(0:5:20), ylims=(0,22), kw...)
    errfield(U; kw...) = plot(contourf(t_plot, KSE.xspan, abs.(U_fom .- U), lw=0, color=:roma);
                              yticks=(0:5:20), ylims=(0,22), colorbar_ticks=(0:0.5:5), clim=(0,5), kw...)
    p_fom = field(U_fom; left_margin=15mm, top_margin=5mm, ylabel=L"\textbf{Full}" * "\n" * L"\omega", xformatter=_->"")
    pblank = plot(axis=false, legend=false, grid=false, foreground_color_subplot=:transparent)
    p_int = field(U_int; left_margin=15mm, ylabel=L"\textbf{Intrusive}" * "\n" * L"\omega", xformatter=_->"")
    p_int_err = errfield(U_int; xformatter=_->"", yformatter=_->"")
    p_LS = field(U_LS; left_margin=15mm, ylabel=L"\textbf{OpInf}" * "\n" * L"\omega", xformatter=_->"")
    p_LS_err = errfield(U_LS; xformatter=_->"", yformatter=_->"")
    p_ephec = field(U_ephec; bottom_margin=15mm, left_margin=15mm,
                    ylabel=L"\textbf{EP}\rm{-}\textbf{OpInf}" * "\n" * L"\omega", xlabel=L"t" * "\n" * L"\textbf{Predicted}~" * L"x(\omega,t)")
    p_ephec_err = errfield(U_ephec; bottom_margin=15mm, xlabel=L"t" * "\n" * L"\textbf{Error}", yformatter=_->"")

    return plot(
        p_fom, pblank,
        p_int, p_int_err,
        p_LS, p_LS_err,
        p_ephec, p_ephec_err,
        fontfamily="Computer Modern", layout=@layout([grid(4,2)]),
        size=(2000, 1080),
        guidefontsize=25, tickfontsize=17, plot_titlefontsize=30,
        plot_title=title * " (r = $(r))",
    )
end
;

In [ ]:
i = num_test_ic ÷ 2  # one of the random test cases (case 25 in the paper)

# Random input within the training region
println("a = $(ic_a_rand_in[i]), b = $(ic_b_rand_in[i])")
p1 = plot_flow_fields(ic_a_rand_in[i], ic_b_rand_in[i], "Predicted Flow Fields and Errors of Test 1: Interpolation")
savefig(p1, "figures/kse_test1_ff.png")
p1

## Test 2

In [ ]:
TEST_RES["test2_AC"], TEST_RES["test2_AC_ERR"], TEST_RES["test2_state_err_short"] = run_test(ic_a_rand_out, ic_b_rand_out)
DataFrame(r=ro; (Symbol(key, "_AC_err") => vec(TEST_RES["test2_AC_ERR"][key]) for key in ac_keys)...)

In [ ]:
DataFrame(r=ro; (Symbol(key, "_state_err_t≤$(Int(T_SHORT))") => vec(TEST_RES["test2_state_err_short"][key]) for key in rom_keys)...)

In [ ]:
save(testresultfile, TEST_RES)

In [ ]:
p = plot_ac4(TEST_RES["test2_AC"], "Test 2: extrapolation")
savefig(p, "figures/kse_test2_ac4.pdf")
p

In [ ]:
p = plot_ac_err(TEST_RES["test2_AC_ERR"], "Test 2: Extrapolation")
savefig(p, "figures/kse_test2_ac_err.pdf")
p

In [ ]:
plot_short_err(TEST_RES["test2_state_err_short"], "Test 2: Extrapolation")

### Flow field of one test case

In [ ]:
i = num_test_ic ÷ 2  # the last case below the training range (case 25 in the paper)

# Random input outside the training region
println("a = $(ic_a_rand_out[i]), b = $(ic_b_rand_out[i])")
p2 = plot_flow_fields(ic_a_rand_out[i], ic_b_rand_out[i], "Predicted Flow Fields and Errors of Test 2: Extrapolation")
savefig(p2, "figures/kse_test2_ff.png")
p2

## Remarks

- The autocorrelation is a statistic of the attractor. ROMs that reproduce it for new initial conditions capture the long-time chaotic dynamics even though their trajectories decorrelate from the FOM.
- The Lyapunov-exponent and Kaplan–Yorke analysis of notebook 06 can also be repeated for the test initial conditions, but it is expensive and is not done here.